In [0]:
%run /Workspace/Users/shreyash270204@outlook.com/databricks/utilities/config.py

Config loaded. STORAGE_ACCOUNT=financestorage1 SNAPSHOT_DATE=latest per exchange TAXONOMY_VERSION=v1


In [0]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

In [0]:
MAPPING_SEED = {
    ("equity", "sector"): {
        "Communication Services": ("Technology", 0.75),
        "Consumer Discretionary": ("Consumer", 0.98),
        "Consumer Staples": ("Consumer", 0.98),
        "Energy": ("Energy", 1.0),
        "Financials": ("Financial Services", 0.98),
        "Health Care": ("Healthcare", 0.98),
        "Industrials": ("Industrials", 1.0),
        "Information Technology": ("Technology", 1.0),
        "Materials": ("Materials", 1.0),
        "Real Estate": ("Real Estate", 1.0),
        "Utilities": ("Utilities", 1.0),
    },
    ("equity", "industry_group"): {
        "Automobiles & Components": ("Consumer", 0.85),
        "Banks": ("Financial Services", 0.98),
        "Capital Goods": ("Industrials", 0.95),
        "Commercial & Professional Services": ("Industrials", 0.85),
        "Consumer Durables & Apparel": ("Consumer", 0.95),
        "Consumer Services": ("Consumer", 0.9),
        "Diversified Financials": ("Financial Services", 0.98),
        "Energy": ("Energy", 1.0),
        "Food & Staples Retailing": ("Consumer", 0.95),
        "Food, Beverage & Tobacco": ("Consumer", 0.95),
        "Health Care Equipment & Services": ("Healthcare", 0.98),
        "Household & Personal Products": ("Consumer", 0.95),
        "Insurance": ("Financial Services", 0.98),
        "Materials": ("Materials", 1.0),
        "Media & Entertainment": ("Technology", 0.7),
        "Pharmaceuticals, Biotechnology & Life Sciences": ("Healthcare", 1.0),
        "Real Estate": ("Real Estate", 1.0),
        "Retailing": ("Consumer", 0.95),
        "Semiconductors & Semiconductor Equipment": ("Technology", 1.0),
        "Software & Services": ("Technology", 1.0),
        "Technology Hardware & Equipment": ("Technology", 1.0),
        "Telecommunication Services": ("Technology", 0.75),
        "Transportation": ("Industrials", 0.95),
        "Utilities": ("Utilities", 1.0),
    },
    ("etf", "category_group"): {
        "Alternatives": (None, 0.3),
        "Cash": (None, 0.2),
        "Commodities": (None, 0.3),
        "Communication Services": ("Technology", 0.75),
        "Consumer Discretionary": ("Consumer", 0.98),
        "Consumer Staples": ("Consumer", 0.98),
        "Currencies": (None, 0.2),
        "Derivatives": (None, 0.2),
        "Energy": ("Energy", 1.0),
        "Equities": (None, 0.2),
        "Financials": ("Financial Services", 0.98),
        "Fixed Income": (None, 0.2),
        "Health Care": ("Healthcare", 0.98),
        "Industrials": ("Industrials", 1.0),
        "Information Technology": ("Technology", 1.0),
        "Materials": ("Materials", 1.0),
        "Real Estate": ("Real Estate", 1.0),
        "Utilities": ("Utilities", 1.0),
    },
    ("etf", "category"): {
        "Blend": (None, 0.1),
        "Cash": (None, 0.1),
        "Commercial Real Estate": ("Real Estate", 0.95),
        "Corporate Bonds": (None, 0.1),
        "Developed Markets": (None, 0.1),
        "Emerging Markets": (None, 0.1),
        "Frontier Markets": (None, 0.1),
        "Government Bonds": (None, 0.1),
        "Growth": (None, 0.1),
        "High Yield Bonds": (None, 0.1),
        "Inflation-Protected Securities": (None, 0.1),
        "Investment Grade Bonds": (None, 0.1),
        "Large Cap": (None, 0.1),
        "Micro Cap": (None, 0.1),
        "Mid Cap": (None, 0.1),
        "Money Market Instruments": (None, 0.1),
        "Municipal Bonds": (None, 0.1),
        "REITs": ("Real Estate", 0.95),
        "Real Estate Development": ("Real Estate", 0.95),
        "Real Estate Services": ("Real Estate", 0.9),
        "Residential Real Estate": ("Real Estate", 0.95),
        "Small Cap": (None, 0.1),
        "Treasury Bonds": (None, 0.1),
        "Value": (None, 0.1),
    },
    ("fund", "category_group"): {
        "Alternative": (None, 0.3),
        "Alternatives": (None, 0.3),
        "Cash": (None, 0.2),
        "Commodities": (None, 0.3),
        "Communication Services": ("Technology", 0.75),
        "Consumer Discretionary": ("Consumer", 0.98),
        "Consumer Staples": ("Consumer", 0.98),
        "Currencies": (None, 0.2),
        "Derivatives": (None, 0.2),
        "Energy": ("Energy", 1.0),
        "Equities": (None, 0.2),
        "Financials": ("Financial Services", 0.98),
        "Fixed Income": (None, 0.2),
        "Health Care": ("Healthcare", 0.98),
        "Industrials": ("Industrials", 1.0),
        "Information Technology": ("Technology", 1.0),
        "Materials": ("Materials", 1.0),
        "Miscellaneous": (None, 0.05),
        "Real Estate": ("Real Estate", 1.0),
        "Utilities": ("Utilities", 1.0),
    },
    ("fund", "category"): {
        "Allocation": (None, 0.1),
        "Alternative": (None, 0.1),
        "Asia": (None, 0.1),
        "Blend": (None, 0.1),
        "Bonds": (None, 0.1),
        "Cash": (None, 0.1),
        "China": (None, 0.1),
        "Commercial Real Estate": ("Real Estate", 0.95),
        "Commodities": (None, 0.3),
        "Commodities Broad Basket": (None, 0.3),
        "Communications": ("Technology", 0.7),
        "Consumer Discretionary": ("Consumer", 0.98),
        "Consumer Staples": ("Consumer", 0.98),
        "Corporate Bonds": (None, 0.1),
        "Currencies": (None, 0.1),
        "Derivatives": (None, 0.1),
        "Developed Markets": (None, 0.1),
        "Emerging Markets": (None, 0.1),
        "Energy": ("Energy", 1.0),
        "Equities": (None, 0.2),
        "Europe": (None, 0.1),
        "Factors": (None, 0.1),
        "Financials": ("Financial Services", 0.98),
        "France": (None, 0.1),
        "Frontier Markets": (None, 0.1),
        "Funds": (None, 0.05),
        "Fundss": (None, 0.05),
        "Germany": (None, 0.1),
        "Government Bonds": (None, 0.1),
        "Growth": (None, 0.1),
        "Health Care": ("Healthcare", 0.98),
        "High Yield Bonds": (None, 0.1),
        "Industrials": ("Industrials", 1.0),
        "Inflation-Protected Securities": (None, 0.1),
        "Investment Grade Bonds": (None, 0.1),
        "Italy": (None, 0.1),
        "Japan": (None, 0.1),
        "Large Cap": (None, 0.1),
        "Loans": (None, 0.1),
        "Materials": ("Materials", 1.0),
        "Micro Cap": (None, 0.1),
        "Mid Cap": (None, 0.1),
        "Miscellaneous": (None, 0.05),
        "Money Market Instruments": (None, 0.1),
        "Municipal Bonds": (None, 0.1),
        "Pension Plans": (None, 0.1),
        "REITs": ("Real Estate", 0.95),
        "Real Estate": ("Real Estate", 1.0),
        "Real Estate Development": ("Real Estate", 0.95),
        "Real Estate Services": ("Real Estate", 0.9),
        "Residential Real Estate": ("Real Estate", 0.95),
        "Scandinavia": (None, 0.1),
        "Sector": (None, 0.05),
        "Small Cap": (None, 0.1),
        "Spain": (None, 0.1),
        "Technology": ("Technology", 1.0),
        "Trading": (None, 0.1),
        "Treasury Bonds": (None, 0.1),
        "United Kingdom": (None, 0.1),
        "United States": (None, 0.1),
        "Utilities": ("Utilities", 1.0),
        "Value": (None, 0.1),
        "World": (None, 0.1),
    },
}

In [0]:
flat_rows = []
for (asset_type, source_field), values in MAPPING_SEED.items():
    for source_value, (theme, confidence) in values.items():
        flat_rows.append((asset_type, source_field, source_value, theme, "RULE", confidence))

schema = T.StructType([
    T.StructField("asset_type", T.StringType(), False),
    T.StructField("source_field", T.StringType(), False),
    T.StructField("source_value", T.StringType(), False),
    T.StructField("normalized_theme", T.StringType(), True),
    T.StructField("mapping_method", T.StringType(), False),
    T.StructField("confidence_score", T.DoubleType(), False),
])

incoming = (
    spark.createDataFrame(flat_rows, schema=schema)
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
    .withColumn(
        "mapping_id",
        F.sha2(F.concat_ws("|", "asset_type", "source_field", "source_value", "taxonomy_version"), 256)
    )
    .withColumn("effective_from", F.current_date())
    .withColumn("effective_to", F.lit(None).cast("date"))
    .withColumn("is_active", F.lit(True))
)

print(f"Built {incoming.count()} mapping rows for taxonomy_version={TAXONOMY_VERSION}")

Built 160 mapping rows for taxonomy_version=v1


In [0]:
target_path = silver_path("taxonomy_mapping")

if DeltaTable.isDeltaTable(spark, target_path):
    target = DeltaTable.forPath(spark, target_path)

    keys = incoming.select("asset_type", "source_field", "source_value").distinct()
    (
        target.alias("t")
        .merge(
            keys.alias("s"),
            "t.asset_type = s.asset_type AND t.source_field = s.source_field "
            "AND t.source_value = s.source_value AND t.is_active = true "
            f"AND t.taxonomy_version != '{TAXONOMY_VERSION}'"
        )
        .whenMatchedUpdate(set={"is_active": "false", "effective_to": "current_date()"})
        .execute()
    )

    (
        target.alias("t")
        .merge(
            incoming.alias("s"),
            "t.asset_type = s.asset_type AND t.source_field = s.source_field "
            "AND t.source_value = s.source_value AND t.taxonomy_version = s.taxonomy_version"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    incoming.write.format("delta").mode("overwrite").save(target_path)

taxonomy_mapping_df = spark.read.format("delta").load(target_path)
print(f"taxonomy_mapping: {taxonomy_mapping_df.count()} total rows ({taxonomy_mapping_df.where('is_active = true').count()} active)")

taxonomy_mapping: 160 total rows (160 active)


In [0]:
active = taxonomy_mapping_df.where("is_active = true")
themed = active.where("normalized_theme IS NOT NULL").count()
not_applicable = active.where("normalized_theme IS NULL").count()
low_confidence = active.where("confidence_score < 0.75").count()

print(f"Themed: {themed}")
print(f"No applicable theme (asset-class/style/geography, not a sector): {not_applicable}")
print(f"Below 0.75 confidence (review required per spec thresholds): {low_confidence}")

display(active.orderBy("asset_type", "source_field", "confidence_score"))

Themed: 78
No applicable theme (asset-class/style/geography, not a sector): 82
Below 0.75 confidence (review required per spec thresholds): 84


asset_type,source_field,source_value,normalized_theme,mapping_method,confidence_score,taxonomy_version,mapping_id,effective_from,effective_to,is_active
equity,industry_group,Media & Entertainment,Technology,RULE,0.7,v1,79d67e02bac7fcda9de41fd257bbd7fd7a12afe5b0af2ae9746494836cf96917,2026-08-21,null,true
equity,industry_group,Telecommunication Services,Technology,RULE,0.75,v1,2bb092bc8535864d974f3353f663aaa3acca499465e0381f57e0759427dabb09,2026-08-21,null,true
equity,industry_group,Commercial & Professional Services,Industrials,RULE,0.85,v1,f0c457fb1483701e171c399723720a810e651089c105c244306d0d7035c5ea13,2026-08-21,null,true
equity,industry_group,Automobiles & Components,Consumer,RULE,0.85,v1,82775ec00d186a2a585a5aeb84db069e692a33a5179e6b0cdb352c1eb6edb600,2026-08-21,null,true
equity,industry_group,Consumer Services,Consumer,RULE,0.9,v1,ab17a1832426da229b0bce3b5afb01faa87283ab1e32d62805faeaba2a6f5a5c,2026-08-21,null,true
equity,industry_group,Capital Goods,Industrials,RULE,0.95,v1,d2e1045089d0097fa392a415a341b3d4554d9cf1d212299b202f1b7995051fd3,2026-08-21,null,true
equity,industry_group,Household & Personal Products,Consumer,RULE,0.95,v1,932d2c3ccd4f871366ac32f59e97897c77a427a642ac71d2ecc78e65bff897da,2026-08-21,null,true
equity,industry_group,Transportation,Industrials,RULE,0.95,v1,ccde90a911752e5af4bad035201addce0065e0e445fa19808587e9b9b540d453,2026-08-21,null,true
equity,industry_group,Consumer Durables & Apparel,Consumer,RULE,0.95,v1,88c957fd7e798aa940eab4028eddcdb22d1359f48e6a2afcea33af04636e3f57,2026-08-21,null,true
equity,industry_group,"Food, Beverage & Tobacco",Consumer,RULE,0.95,v1,c154dc77c6421ab99407a5629f9a7bef5bdc161fa216d993376aec0ec9231863,2026-08-21,null,true


In [0]:
dbutils.fs.ls("abfss://quarantine@financestorage1.dfs.core.windows.net/")

[FileInfo(path='abfss://quarantine@financestorage1.dfs.core.windows.net/invalid_records/', name='invalid_records/', size=0, modificationTime=1787310181000)]